# Simulación balística Monte Carlo interactiva

Este cuaderno permite estudiar un modelo balístico con rozamiento lineal y parámetros inciertos.

La idea es ejecutar el simulador cambiando los parámetros mediante controles deslizantes, sin necesidad de modificar el código.

El modelo determinista empleado es

$$
\ddot{x}=-\lambda \dot{x}, \qquad
\ddot{y}=-\lambda \dot{y}-g,
$$

donde

$$
\lambda=\frac{c}{m}.
$$

La trayectoria viene dada por

$$
x(t)=\frac{v_0\cos\theta}{\lambda}\left(1-e^{-\lambda t}\right),
$$

$$
y(t)=
\frac{v_0\sin\theta+g/\lambda}{\lambda}
\left(1-e^{-\lambda t}\right)
-\frac{g}{\lambda}t.
$$

El tiempo de vuelo $T_f$ se obtiene resolviendo numéricamente

$$
y(T_f)=0,
$$

y el alcance es

$$
R=x(T_f).
$$

## 1. Inicialización

Ejecute la celda siguiente. El código queda oculto para que el cuaderno se comporte como una pequeña aplicación interactiva.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.optimize import brentq
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True


def y_posicion(t, v0, theta_rad, c, m, g=9.81):
    """Altura y(t) para rozamiento lineal."""
    lam = c / m
    if lam <= 0:
        return v0*np.sin(theta_rad)*t - 0.5*g*t**2
    return ((v0*np.sin(theta_rad) + g/lam)/lam)*(1 - np.exp(-lam*t)) - (g/lam)*t


def x_posicion(t, v0, theta_rad, c, m):
    """Posición horizontal x(t) para rozamiento lineal."""
    lam = c / m
    if lam <= 0:
        return v0*np.cos(theta_rad)*t
    return (v0*np.cos(theta_rad)/lam)*(1 - np.exp(-lam*t))


def tiempo_vuelo(v0, theta_rad, c, m, g=9.81):
    """Calcula el tiempo de vuelo resolviendo y(t)=0."""
    if v0 <= 0 or m <= 0 or c < 0:
        return np.nan

    t_sup = max(1.0, 2*v0*np.sin(theta_rad)/g + 10.0)

    def f(t):
        return y_posicion(t, v0, theta_rad, c, m, g)

    eps = 1e-9
    while f(t_sup) > 0 and t_sup < 1e5:
        t_sup *= 2

    try:
        return brentq(f, eps, t_sup, maxiter=100)
    except Exception:
        return np.nan


def alcance(v0, theta_deg, c, m, g=9.81):
    """Calcula el alcance horizontal."""
    theta_rad = np.deg2rad(theta_deg)
    tf = tiempo_vuelo(v0, theta_rad, c, m, g)
    if not np.isfinite(tf):
        return np.nan
    return x_posicion(tf, v0, theta_rad, c, m)


def trayectoria(v0, theta_deg, c, m, g=9.81, n=300):
    """Devuelve puntos de la trayectoria."""
    theta_rad = np.deg2rad(theta_deg)
    tf = tiempo_vuelo(v0, theta_rad, c, m, g)
    if not np.isfinite(tf):
        return np.array([]), np.array([]), np.nan
    t = np.linspace(0, tf, n)
    x = x_posicion(t, v0, theta_rad, c, m)
    y = y_posicion(t, v0, theta_rad, c, m, g)
    return x, y, tf


def simular_montecarlo(
    N=5000,
    v0_media=300.0,
    v0_sigma=5.0,
    theta_media=45.0,
    theta_sigma=1.0,
    c_media=0.08,
    c_sigma=0.01,
    m_media=10.0,
    m_sigma=0.2,
    blanco=5000.0,
    tolerancia=100.0,
    semilla=123
):
    """Simulación Monte Carlo del alcance balístico."""
    rng = np.random.default_rng(semilla)

    V0 = rng.normal(v0_media, v0_sigma, N)
    TH = rng.normal(theta_media, theta_sigma, N)
    C = rng.normal(c_media, c_sigma, N)
    M = rng.normal(m_media, m_sigma, N)

    V0 = np.maximum(V0, 1e-6)
    TH = np.clip(TH, 1e-3, 89.999)
    C = np.maximum(C, 1e-9)
    M = np.maximum(M, 1e-6)

    R = np.empty(N)
    for i in range(N):
        R[i] = alcance(V0[i], TH[i], C[i], M[i])

    R = R[np.isfinite(R)]

    media = np.mean(R)
    desviacion = np.std(R, ddof=1)
    error_estandar = desviacion / np.sqrt(len(R))
    ic95 = (media - 1.96*error_estandar, media + 1.96*error_estandar)

    impactos = np.abs(R - blanco) <= tolerancia
    p_impacto = np.mean(impactos)

    return {
        "R": R,
        "media": media,
        "desviacion": desviacion,
        "error_estandar": error_estandar,
        "ic95": ic95,
        "p_impacto": p_impacto,
        "n_validos": len(R)
    }

## 2. Trayectoria determinista

Modifique los controles para observar cómo cambian la trayectoria, el tiempo de vuelo y el alcance.

In [ ]:
def app_trayectoria(v0, theta, c, m):
    x, y, tf = trayectoria(v0, theta, c, m)
    R = alcance(v0, theta, c, m)

    fig, ax = plt.subplots()
    if len(x) > 0:
        ax.plot(x, y)
        ax.set_xlabel("Alcance horizontal x(t) [m]")
        ax.set_ylabel("Altura y(t) [m]")
        ax.set_title("Trayectoria balística con rozamiento lineal")
        ax.set_ylim(bottom=0)
        ax.axhline(0, linewidth=1)
    else:
        ax.text(0.5, 0.5, "No se pudo calcular la trayectoria", ha="center", va="center")

    plt.tight_layout()
    plt.show()

    texto = f"""
**Resultados deterministas**

- Tiempo de vuelo: `{tf:.3f}` s  
- Alcance: `{R:.3f}` m  
- Parámetro de rozamiento: `lambda = c/m = {c/m:.5f}` s$^{{-1}}$
"""
    display(Markdown(texto))

widgets.interact(
    app_trayectoria,
    v0=widgets.FloatSlider(value=300, min=50, max=800, step=10, description="v0 [m/s]"),
    theta=widgets.FloatSlider(value=45, min=5, max=85, step=1, description="theta [deg]"),
    c=widgets.FloatSlider(value=0.08, min=0.001, max=0.5, step=0.005, description="c"),
    m=widgets.FloatSlider(value=10, min=1, max=50, step=0.5, description="m [kg]")
);

## 3. Simulación Monte Carlo interactiva

Ahora los parámetros iniciales se consideran variables aleatorias.

Se supone, como modelo básico, que

$$
V_0\sim N(\mu_{V_0},\sigma_{V_0}^2),
\qquad
\Theta_0\sim N(\mu_{\Theta},\sigma_{\Theta}^2),
$$

$$
C\sim N(\mu_C,\sigma_C^2),
\qquad
M\sim N(\mu_M,\sigma_M^2).
$$

El objetivo es estimar:

- el alcance medio;
- la desviación típica del alcance;
- el intervalo de confianza al 95%;
- la probabilidad de impacto en un blanco.

In [ ]:
def app_montecarlo(
    N,
    v0_media,
    v0_sigma,
    theta_media,
    theta_sigma,
    c_media,
    c_sigma,
    m_media,
    m_sigma,
    blanco,
    tolerancia,
    semilla
):
    res = simular_montecarlo(
        N=N,
        v0_media=v0_media,
        v0_sigma=v0_sigma,
        theta_media=theta_media,
        theta_sigma=theta_sigma,
        c_media=c_media,
        c_sigma=c_sigma,
        m_media=m_media,
        m_sigma=m_sigma,
        blanco=blanco,
        tolerancia=tolerancia,
        semilla=semilla
    )
    R = res["R"]

    fig, ax = plt.subplots()
    ax.hist(R, bins=40, density=True, alpha=0.8)
    ax.axvline(res["media"], linestyle="--", label="Media")
    ax.axvline(blanco, linestyle="-", label="Blanco")
    ax.axvspan(blanco - tolerancia, blanco + tolerancia, alpha=0.2, label="Zona de impacto")
    ax.set_xlabel("Alcance R [m]")
    ax.set_ylabel("Densidad aproximada")
    ax.set_title("Distribución simulada del alcance")
    ax.legend()
    plt.tight_layout()
    plt.show()

    texto = f"""
**Resultados Monte Carlo**

- Simulaciones válidas: `{res["n_validos"]}`  
- Alcance medio: `{res["media"]:.3f}` m  
- Desviación típica: `{res["desviacion"]:.3f}` m  
- Error estándar: `{res["error_estandar"]:.3f}` m  
- IC95% para el alcance medio: `[{res["ic95"][0]:.3f}, {res["ic95"][1]:.3f}]` m  
- Probabilidad estimada de impacto: `{res["p_impacto"]:.4f}`  
"""
    display(Markdown(texto))

widgets.interact(
    app_montecarlo,
    N=widgets.IntSlider(value=3000, min=500, max=20000, step=500, description="N"),
    v0_media=widgets.FloatSlider(value=300, min=50, max=800, step=10, description="mu v0"),
    v0_sigma=widgets.FloatSlider(value=5, min=0.1, max=50, step=0.5, description="sigma v0"),
    theta_media=widgets.FloatSlider(value=45, min=5, max=85, step=1, description="mu theta"),
    theta_sigma=widgets.FloatSlider(value=1, min=0.1, max=10, step=0.1, description="sigma theta"),
    c_media=widgets.FloatSlider(value=0.08, min=0.001, max=0.5, step=0.005, description="mu c"),
    c_sigma=widgets.FloatSlider(value=0.01, min=0.0001, max=0.1, step=0.001, description="sigma c"),
    m_media=widgets.FloatSlider(value=10, min=1, max=50, step=0.5, description="mu m"),
    m_sigma=widgets.FloatSlider(value=0.2, min=0.01, max=5, step=0.05, description="sigma m"),
    blanco=widgets.FloatSlider(value=5000, min=500, max=20000, step=100, description="blanco"),
    tolerancia=widgets.FloatSlider(value=100, min=10, max=1000, step=10, description="Delta"),
    semilla=widgets.IntText(value=123, description="semilla")
);

## 4. Botón de simulación controlada

En algunos ordenadores, mover un deslizador puede recalcular demasiado a menudo. La siguiente versión sólo ejecuta la simulación al pulsar un botón.

In [ ]:
N_w = widgets.IntSlider(value=5000, min=500, max=50000, step=500, description="N")
v0_w = widgets.FloatSlider(value=300, min=50, max=800, step=10, description="mu v0")
sv0_w = widgets.FloatSlider(value=5, min=0.1, max=50, step=0.5, description="sigma v0")
th_w = widgets.FloatSlider(value=45, min=5, max=85, step=1, description="mu theta")
sth_w = widgets.FloatSlider(value=1, min=0.1, max=10, step=0.1, description="sigma theta")
c_w = widgets.FloatSlider(value=0.08, min=0.001, max=0.5, step=0.005, description="mu c")
sc_w = widgets.FloatSlider(value=0.01, min=0.0001, max=0.1, step=0.001, description="sigma c")
m_w = widgets.FloatSlider(value=10, min=1, max=50, step=0.5, description="mu m")
sm_w = widgets.FloatSlider(value=0.2, min=0.01, max=5, step=0.05, description="sigma m")
blanco_w = widgets.FloatSlider(value=5000, min=500, max=20000, step=100, description="blanco")
delta_w = widgets.FloatSlider(value=100, min=10, max=1000, step=10, description="Delta")
semilla_w = widgets.IntText(value=123, description="semilla")
boton = widgets.Button(description="Ejecutar simulación", button_style="success")
salida = widgets.Output()

def ejecutar_simulacion(_):
    with salida:
        clear_output(wait=True)
        app_montecarlo(
            N_w.value, v0_w.value, sv0_w.value, th_w.value, sth_w.value,
            c_w.value, sc_w.value, m_w.value, sm_w.value,
            blanco_w.value, delta_w.value, semilla_w.value
        )

boton.on_click(ejecutar_simulacion)

display(widgets.VBox([
    widgets.HTML("<b>Parámetros principales</b>"),
    N_w, v0_w, sv0_w, th_w, sth_w,
    widgets.HTML("<b>Rozamiento y masa</b>"),
    c_w, sc_w, m_w, sm_w,
    widgets.HTML("<b>Blanco</b>"),
    blanco_w, delta_w, semilla_w,
    boton,
    salida
]))

## 5. Observaciones didácticas

Este cuaderno está pensado para usarse como una aplicación interactiva:

1. Ejecute todas las celdas con `Run All`.
2. Modifique los controles.
3. Observe cómo cambian la trayectoria, el histograma del alcance y la probabilidad de impacto.

La parte esencial del método de Monte Carlo es que la salida ya no es un único número determinista, sino una distribución de resultados posibles.